# Feature Extraction Demo — GOAL Framework

This notebook demonstrates the **feature extraction utilities** in GOAL.  These
tools let you inspect, slice, and compose intermediate representations from any
equivariant backbone — useful for building downstream models, debugging
architecture choices, and exploring what the network learns at each layer.

**What this notebook covers:**
1. Building an `AtomicGraph` from inline ASE structures (no files needed)
2. Loading a MACE foundation model and inspecting its layer structure
3. Extracting features at individual layers and across all layers (multi-scale)
4. Extracting invariant scalar channels for standard MLPs
5. Freezing a backbone for pure feature-extraction mode
6. Pooling node features to graph-level representations
7. Wiring a frozen backbone into a simple downstream training loop

In [ ]:
from __future__ import annotations

import typing

import torch
import torch.nn as nn
from ase import Atoms
from torch_geometric.data import Batch

from goal.ml.data.graph import AtomicGraph, NodeFeatures
from goal.ml.utils.extraction import (
    HookBasedExtractor,
    LayerBackbone,
    MultiScaleBackbone,
    FrozenBackbone,
    extract_scalars,
    extract_irrep_channels,
    pool_nodes,
    describe_irreps,
)

## 1. Build AtomicGraphs from Inline Structures

We create water and ethanol molecules directly in code — no data files required.
Each is converted to an `AtomicGraph` using the ASE boundary, then batched with
PyG's `Batch.from_data_list()` for downstream processing.

In [ ]:
# Water molecule
atoms_water: Atoms = Atoms(
    symbols="OH2",
    positions=[[0.0, 0.0, 0.0], [0.96, 0.0, 0.0], [-0.24, 0.93, 0.0]],
)

# Ethanol molecule (rough geometry)
atoms_ethanol: Atoms = Atoms(
    symbols="C2H6O",
    positions=[
        [-0.74, -0.05, 0.0],   # C
        [0.74, 0.05, 0.0],     # C
        [-1.16, 0.93, 0.35],   # H
        [-1.16, -0.25, -0.99], # H
        [-1.16, -0.79, 0.68],  # H
        [1.16, 0.79, -0.68],   # H
        [1.16, -0.93, -0.35],  # H
        [1.16, 0.25, 0.99],    # H
        [1.90, 0.05, 0.0],     # O
    ],
)

cutoff: float = 5.0
graph_water: AtomicGraph = AtomicGraph.from_ase(atoms_water, cutoff=cutoff)
graph_ethanol: AtomicGraph = AtomicGraph.from_ase(atoms_ethanol, cutoff=cutoff)

# Batch both structures together
batched_graph: Batch = Batch.from_data_list([graph_water, graph_ethanol])

print(f"Water:   {graph_water.num_atoms} atoms, {graph_water.edge_index.shape[1]} edges")
print(f"Ethanol: {graph_ethanol.num_atoms} atoms, {graph_ethanol.edge_index.shape[1]} edges")
print(f"Batch:   {batched_graph.num_nodes} atoms total, {batched_graph.num_graphs} graphs")

## 2. Load MACE and Inspect Layer Structure

Load the pre-trained MACE-MP foundation model. MACE models have several
interaction blocks — each one refines the node features through message passing.
Earlier layers capture local geometry; later layers integrate information from
a larger receptive field.

> **Note:** This cell requires `mace-torch` to be installed.  
> Install with `pip install goal[mace]` or `pip install goal[mace]`.

In [ ]:
from goal.ml.adapters.mace import MACEAdapter

adapter: MACEAdapter = MACEAdapter.from_pretrained("large")

n_layers: int = adapter.num_layers
print(f"MACE-large has {n_layers} interaction layers\n")

for i in range(n_layers):
    layer_irreps: str = str(adapter.irreps_at_layer(i))
    print(f"  Layer {i}:")
    describe_irreps(layer_irreps)

## 3. Extract Features at Different Layers

Use `extract_features()` to capture the intermediate node representations at
any interaction layer.  Layer `0` is the first message-passing step (most local),
and layer `N-1` is the last (most global).

You can also pass `layer='all'` to concatenate features from every layer — useful
for multi-scale models that need both local and global information simultaneously.

In [ ]:
# Extract features at each interaction layer
for layer_idx in range(n_layers):
    feats: NodeFeatures = adapter.extract_features(graph_water, layer=layer_idx)
    print(f"Layer {layer_idx}: shape={feats.node_feats.shape}  irreps={feats.irreps}")

# Extract final-layer features (default)
final_feats: NodeFeatures = adapter.extract_features(graph_water, layer=-1)
print(f"\nFinal layer (layer=-1): {final_feats.node_feats.shape}")

# Extract multi-scale features (all layers concatenated)
all_feats: NodeFeatures = adapter.extract_features(graph_water, layer="all")
print(f"All layers concatenated: {all_feats.node_feats.shape}  irreps={all_feats.irreps}")

## 4. Extract Scalar (Invariant) Channels

Equivariant features contain L=0 (scalar), L=1 (vector), L=2 (matrix), and
possibly higher-order components.  Standard MLPs **cannot** process the vector
and tensor components — they are not invariant to rotation.

`extract_scalars()` slices out only the L=0 channels, giving you a rotation-
invariant representation that can be safely fed to any MLP, DeepSet, or
standard neural network layer.

In [ ]:
# Extract scalars (L=0 only) from the final layer
scalars: torch.Tensor = extract_scalars(final_feats)
print(f"Full features shape:  {final_feats.node_feats.shape}")
print(f"Scalar-only shape:    {scalars.shape}")

# Compare with manually checking the irreps breakdown
describe_irreps(final_feats.irreps)

# Extract L=1 vector channels for comparison
vectors: torch.Tensor = extract_irrep_channels(final_feats, l=1)
print(f"\nL=1 vector channels shape: {vectors.shape}")
print(f"  (each L=1 irrep contributes 3 components per multiplicity)")

## 5. Multi-Scale and Frozen Backbone Wrappers

The **composable wrappers** let you build sophisticated feature extraction
pipelines from simple building blocks:

| Wrapper | Effect |
|---|---|
| `LayerBackbone(adapter, layer=k)` | Expose layer *k* as backbone output |
| `MultiScaleBackbone(adapter)` | Concatenate all layers |
| `FrozenBackbone(backbone)` | Freeze all parameters — pure inference |

These wrappers satisfy the `EquivariantBackbone` protocol, so they plug
directly into any `GOALModule`.

In [ ]:
# Multi-scale: concatenate all layers
ms_backbone: MultiScaleBackbone = MultiScaleBackbone(adapter)
print("Multi-scale output irreps:")
describe_irreps(str(ms_backbone.irreps_out))

ms_feats: NodeFeatures = ms_backbone.forward(graph_water)
print(f"\nMulti-scale features shape: {ms_feats.node_feats.shape}")

# Single layer: expose just layer 0 (most local)
layer0_backbone: LayerBackbone = LayerBackbone(adapter, layer=0)
layer0_feats: NodeFeatures = layer0_backbone.forward(graph_water)
print(f"Layer-0 features shape:    {layer0_feats.node_feats.shape}")
print(f"Layer-0 irreps:            {layer0_backbone.irreps_out}")

# Frozen backbone: no gradients flow through
frozen: FrozenBackbone = FrozenBackbone(ms_backbone)
frozen_feats: NodeFeatures = frozen.forward(graph_water)
print(f"\nFrozen features shape:     {frozen_feats.node_feats.shape}")

# Verify all parameters are frozen
n_frozen: int = sum(1 for p in frozen.parameters() if not p.requires_grad)
n_total: int = sum(1 for p in frozen.parameters())
print(f"Frozen parameters: {n_frozen}/{n_total}")

## 6. Pool Node Features to Graph-Level Vectors

`pool_nodes()` reduces per-node features `(N, D)` to per-graph features
`(B, D)` using scatter operations.  Three modes are supported:

- **sum** — equivariance-preserving, used in most GNN readout layers
- **mean** — equivariance-preserving, normalises for graph size
- **max** — **not** equivariant, but can capture salient node features

For equivariant downstream models, always use `sum` or `mean`.

In [ ]:
# Extract features on the batched graph
batch_feats: NodeFeatures = adapter.extract_features(batched_graph, layer=-1)
print(f"Batched node features: {batch_feats.node_feats.shape}")

# Pool to graph-level
graph_sum: torch.Tensor = pool_nodes(
    batch_feats, batched_graph.batch, batched_graph.num_graphs, mode="sum",
)
graph_mean: torch.Tensor = pool_nodes(
    batch_feats, batched_graph.batch, batched_graph.num_graphs, mode="mean",
)

print(f"Sum-pooled graph features:  {graph_sum.shape}")
print(f"Mean-pooled graph features: {graph_mean.shape}")
print(f"\n  Graph 0 (water):   sum norm = {graph_sum[0].norm():.4f}")
print(f"  Graph 1 (ethanol): sum norm = {graph_sum[1].norm():.4f}")

## 7. Downstream Training Loop — DeepSet on Frozen Features

Put it all together: use a **frozen** multi-scale backbone to extract invariant
scalar features, pool them to graph-level vectors, then train a simple MLP
(DeepSet-style) to predict a dummy energy target.

This pattern is common in molecular property prediction — the foundation model
(MACE) acts as a fixed feature extractor and only a lightweight head is trained.
This is much faster and requires far less data than fine-tuning the entire model.

In [ ]:
# --- Frozen backbone setup ---
frozen_ms: FrozenBackbone = FrozenBackbone(MultiScaleBackbone(adapter))

# Compute scalar dimension from frozen backbone's irreps
from e3nn.o3 import Irreps

ms_irreps: Irreps = frozen_ms.irreps_out
scalar_dim: int = sum(mul for mul, ir in ms_irreps if ir.l == 0)
print(f"Scalar input dimension: {scalar_dim}")

# --- Simple MLP head ---
head: nn.Sequential = nn.Sequential(
    nn.Linear(scalar_dim, 128),
    nn.SiLU(),
    nn.Linear(128, 64),
    nn.SiLU(),
    nn.Linear(64, 1),
)

# --- Dummy targets ---
target_energies: torch.Tensor = torch.tensor([-76.0, -154.0])  # kcal/mol ish

# --- Training loop ---
optimiser: torch.optim.Adam = torch.optim.Adam(head.parameters(), lr=1e-3)
loss_fn: nn.MSELoss = nn.MSELoss()

for step in range(5):
    optimiser.zero_grad()

    # Extract frozen features → scalars → pool → predict
    feats: NodeFeatures = frozen_ms.forward(batched_graph)
    s: torch.Tensor = extract_scalars(feats)
    graph_repr: torch.Tensor = pool_nodes(
        NodeFeatures(node_feats=s, irreps=f"{scalar_dim}x0e"),
        batched_graph.batch,
        batched_graph.num_graphs,
        mode="sum",
    )
    pred: torch.Tensor = head(graph_repr).squeeze(-1)
    loss: torch.Tensor = loss_fn(pred, target_energies)

    loss.backward()
    optimiser.step()

    print(f"Step {step}: loss={loss.item():.4f}  pred={pred.detach().tolist()}")

print("\nDone — only the MLP head was trained; backbone weights stayed frozen.")